# MLP Superposition Analysis

Detect and characterize superposition in MLP weights: interference,
capacity, orthogonality, and cross-layer summary.

## Why This Matters

MLP superposition provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_superposition_analysis import (
    mlp_input_interference, mlp_output_interference,
    mlp_feature_capacity, mlp_neuron_orthogonality,
    mlp_superposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Input Interference

How much do MLP input (reading) directions interfere with each other?
High interference means features are stored in superposition.

In [ ]:
for layer in range(4):
    result = mlp_input_interference(model, layer=layer)
    flag = ' [SUPERPOSITION]' if result['has_superposition'] else ''
    print(f"  Layer {layer}: mean={result['mean_interference']:.4f}, "
          f"max={result['max_interference']:.4f}, "
          f"high_pairs={result['n_high_interference_pairs']}{flag}")

## Output Interference

How much do output (writing) directions overlap?

In [ ]:
for layer in range(4):
    result = mlp_output_interference(model, layer=layer)
    flag = ' [SUPERPOSITION]' if result['has_superposition'] else ''
    print(f"  Layer {layer}: mean={result['mean_interference']:.4f}, "
          f"max={result['max_interference']:.4f}, "
          f"high_pairs={result['n_high_interference_pairs']}{flag}")

## Feature Capacity

How many effective features can each MLP store vs its dimensionality?

In [ ]:
for layer in range(4):
    result = mlp_feature_capacity(model, layer=layer)
    print(f"  Layer {layer}: d_mlp={result['d_mlp']}, d_model={result['d_model']}, "
          f"ratio={result['capacity_ratio']:.1f}x, "
          f"eff_rank={result['effective_rank']:.2f}, "
          f"utilization={result['rank_utilization']:.3f}")

## Neuron Orthogonality

Are neuron directions approximately orthogonal (no superposition)
or do they have significant overlap?

In [ ]:
for layer in range(4):
    result = mlp_neuron_orthogonality(model, layer=layer, sample_size=30)
    orth = 'Yes' if result['is_approximately_orthogonal'] else 'No'
    print(f"  Layer {layer}: in_overlap={result['input_mean_overlap']:.4f}, "
          f"out_overlap={result['output_mean_overlap']:.4f}, "
          f"orthogonal={orth}")

## Superposition Summary

Cross-layer overview of superposition levels.

In [ ]:
result = mlp_superposition_summary(model)
for p in result['per_layer']:
    level = p['superposition_level'].upper()
    bar = {'low': '▪', 'moderate': '▪▪▪', 'high': '▪▪▪▪▪▪'}[p['superposition_level']]
    print(f"  Layer {p['layer']}: [{level:8s}] ratio={p['capacity_ratio']:.1f}x, "
          f"interference={p['mean_interference']:.4f}, "
          f"eff_rank={p['effective_rank']:.2f} {bar}")